In [1]:
import os
import re
import time
from datetime import datetime

import pandas as pd
import plotly.express as px
import requests
from bs4 import BeautifulSoup
from dash import Dash, Input, Output, dcc, html

os.makedirs("data", exist_ok=True)

In [3]:
import urllib.robotparser

HEADERS = {"User-Agent": "Mozilla/5.0 (student project: web scraping and dash course)"}

pages = {
    "BBC league table": ("https://www.bbc.com", "/sport/football/premier-league/table"),
    "BBC top scorers": ("https://www.bbc.com", "/sport/football/premier-league/top-scorers"),
    "ESPN standings": ("https://www.espn.com", "/soccer/standings/_/league/eng.1"),
}

for name, (site, path) in pages.items():
    try:
        r = requests.get(site + "/robots.txt", headers=HEADERS, timeout=10)
        robots = urllib.robotparser.RobotFileParser()
        robots.parse(r.text.splitlines())
        print(f"{name:20} allowed: {robots.can_fetch('*', site + path)}")
    except Exception as error:
        print(f"{name:20} could not check ->", type(error).__name__)

BBC league table     allowed: True
BBC top scorers      allowed: True
ESPN standings       allowed: True


In [4]:
HEADERS = {"User-Agent": "Mozilla/5.0 (student project: web scraping and dash course)"}
TABLE_URL = "https://www.bbc.com/sport/football/premier-league/table"

response = requests.get(TABLE_URL, headers=HEADERS, timeout=15)
response.encoding = "utf-8"
print("Status:", response.status_code, "(200 = OK) | page size:", len(response.text), "characters")

soup = BeautifulSoup(response.text, "lxml")
print("Page title:", soup.title.text)

rows = soup.select("table tr")
print("Rows found:", len(rows))

for row in rows[:3]:
    print([cell.get_text(strip=True) for cell in row.select("th, td")])

Status: 200 (200 = OK) | page size: 260846 characters
Page title: Premier League Table and Standings - Football - BBC Sport
Rows found: 21
['Team', 'Played', 'Won', 'Drawn', 'Lost', 'Goals For', 'Goals Against', 'Goal Difference', 'Points', 'Form, Last 6 games, Oldest first']
['1Manchester City', '5', '5', '0', '0', '13', '5', '8', '15', 'No ResultWResult WinWResult WinWResult WinWResult WinWResult Win']
['2Leeds United', '6', '3', '3', '0', '8', '3', '5', '12', 'No ResultWResult WinDResult DrawDResult DrawWResult WinDResult Draw']


In [6]:
def is_number(text):
    return re.fullmatch(r"[+-]?\d+", text.replace("\u2212", "-")) is not None

def to_int(text):
    return int(text.replace("\u2212", "-").replace("+", ""))

def get_soup(url, timeout=15):
    last_error = None
    for attempt in range(2):
        try:
            response = requests.get(url, headers=HEADERS, timeout=timeout)
            response.raise_for_status()
            response.encoding = "utf-8"
            return BeautifulSoup(response.text, "lxml")
        except requests.RequestException as error:
            last_error = error
            time.sleep(1)
    raise last_error


def read_rows(url):
    soup = get_soup(url)
    return [[cell.get_text(strip=True) for cell in row.select("th, td")] for row in soup.select("table tr")]

In [7]:
def scrape_bbc_table():
    try:
        data = []
        for cells in read_rows(TABLE_URL):
            if len(cells) < 9:
                continue
            match = re.match(r"^(\d+)(.+)$", cells[0])
            stats = cells[1:9]
            if match is None or not all(is_number(c) for c in stats):
                continue
            p, w, d, l, gf, ga, gd, pts = [to_int(c) for c in stats]
            data.append({"position": int(match.group(1)), "team": match.group(2).strip(), "played": p, "won": w,
                         "drawn": d, "lost": l, "goals_for": gf, "goals_against": ga, "goal_diff": gd, "points": pts})
        table = pd.DataFrame(data)
        if table.empty:
            raise ValueError("no rows found")
    except Exception as error:
        print("Could not read the table ->", type(error).__name__, error)
        return None
    table["scraped_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    table["source"] = "BBC"
    return table


ESPN_URL = "https://www.espn.com/soccer/standings/_/league/eng.1"

def scrape_espn_table():
    try:
        soup = get_soup(ESPN_URL)
        tables = soup.select("table")
        names = [max((a.get_text(strip=True) for a in row.select("a")), key=len) for row in tables[0].select("tr") if row.select("a")]
        stats = []
        for row in tables[1].select("tr"):
            numbers = [to_int(c.get_text(strip=True)) for c in row.select("td") if is_number(c.get_text(strip=True))]
            if len(numbers) >= 8:
                stats.append(numbers[:8])
        data = []
        for position, (name, (p, w, d, l, gf, ga, gd, pts)) in enumerate(zip(names, stats), start=1):
            data.append({"position": position, "team": name, "played": p, "won": w, "drawn": d, "lost": l,
                         "goals_for": gf, "goals_against": ga, "goal_diff": gd, "points": pts})
        table = pd.DataFrame(data)
        if table.empty:
            raise ValueError("no rows found")
    except Exception as error:
        print("ESPN: could not read the table ->", type(error).__name__, error)
        return None
    table["scraped_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    table["source"] = "ESPN"
    return table


def scrape_table():
    table = scrape_bbc_table()
    if table is None:
        print("BBC failed, trying ESPN...")
        table = scrape_espn_table()
    return table


SCORERS_URL = "https://www.bbc.com/sport/football/premier-league/top-scorers"

def scrape_scorers():
    try:
        soup = get_soup(SCORERS_URL)
        rows, goals_col = [], None
        for page_table in soup.select("table"):
            table_rows = page_table.select("tr")
            for row in table_rows:
                labels = [c.get_text(strip=True).lower() for c in row.select("th, td")]
                for i, label in enumerate(labels):
                    if label.startswith("goals") and "per" not in label:
                        goals_col = i
                        break
                if goals_col is not None:
                    break
            if goals_col is not None:
                rows = table_rows
                break
        print("Scorers table: goals column =", goals_col, "| rows =", len(rows))
        data = []
        for row in rows:
            cells = row.select("th, td")
            texts = [c.get_text(strip=True) for c in cells]
            words = []
            for c in cells:
                for t in c.stripped_strings:
                    if not is_number(t) and not re.fullmatch(r"[=T]?\d+", t) and t not in words:
                        words.append(t)
            if len(words) < 1:
                continue
            if goals_col is not None and goals_col < len(texts) and is_number(texts[goals_col]):
                goals = to_int(texts[goals_col])
            else:
                numbers = [to_int(t) for t in texts if is_number(t)]
                if not numbers:
                    continue
                goals = numbers[1] if len(numbers) > 1 else numbers[0]
            data.append({"player": words[0], "team": words[1] if len(words) > 1 else "", "goals": goals})
        scorers = pd.DataFrame(data).drop_duplicates(["player", "team"])
        if scorers.empty:
            raise ValueError("no rows found")
    except Exception as error:
        print("Could not read the scorers ->", type(error).__name__, error)
        return None
    scorers["scraped_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return scorers

In [8]:
table = scrape_table()
time.sleep(1)
scorers = scrape_scorers()

if table is not None:
    table.to_csv("data/pl_table.csv", index=False)
if scorers is not None:
    scorers.to_csv("data/pl_scorers.csv", index=False)

table

Scorers table: goals column = 2 | rows = 30


,position,team,played,won,drawn,lost,goals_for,goals_against,goal_diff,points,scraped_at,source
0,1,Manchester City,5,5,0,0,13,5,8,15,2026-10-10 15:48:59,BBC
1,2,Leeds United,6,3,3,0,8,3,5,12,2026-10-10 15:48:59,BBC
2,3,Arsenal,6,4,0,2,8,5,3,12,2026-10-10 15:48:59,BBC
3,4,Brighton & Hove Albion,5,3,1,1,16,5,11,10,2026-10-10 15:48:59,BBC
4,5,Brentford,5,2,3,0,10,4,6,9,2026-10-10 15:48:59,BBC
5,6,Liverpool,5,2,3,0,7,4,3,9,2026-10-10 15:48:59,BBC
6,7,Everton,5,2,3,0,6,3,3,9,2026-10-10 15:48:59,BBC
7,8,Hull City,5,2,2,1,6,4,2,8,2026-10-10 15:48:59,BBC
8,9,Newcastle United,5,2,2,1,9,9,0,8,2026-10-10 15:48:59,BBC
9,10,Chelsea,5,2,1,2,10,12,-2,7,2026-10-10 15:48:59,BBC


In [9]:
def load_data():
    table = pd.read_csv("data/pl_table.csv")
    scorers = pd.read_csv("data/pl_scorers.csv") if os.path.exists("data/pl_scorers.csv") else None
    table = table.dropna(subset=["team"])
    table["goals_per_match"] = (table["goals_for"] / table["played"]).round(2)
    table["points_per_match"] = (table["points"] / table["played"]).round(2)
    return table, scorers


table, scorers = load_data()
print(table.shape[0], "teams |", 0 if scorers is None else scorers.shape[0], "scorers")
print(table.isna().sum().sum(), "missing values")
table.describe().round(2)

20 teams | 29 scorers
0 missing values


,position,played,won,drawn,lost,goals_for,goals_against,goal_diff,points,goals_per_match,points_per_match
count,20.00,20.00,20.00,20.00,20.00,20.00,20.00,20.0,20.00,20.00,20.00
mean,10.50,5.10,1.75,1.60,1.75,7.10,7.10,0.0,6.85,1.39,1.33
std,5.92,0.31,1.29,1.10,1.29,3.48,2.92,5.1,3.66,0.69,0.68
min,1.00,5.00,0.00,0.00,0.00,1.00,3.00,-9.0,2.00,0.20,0.40
25%,5.75,5.00,1.00,1.00,0.75,5.75,4.75,-4.0,4.00,1.15,0.80
50%,10.50,5.00,2.00,2.00,2.00,6.50,8.00,-0.5,6.50,1.27,1.30
75%,15.25,5.00,2.00,2.25,3.00,8.25,9.25,3.0,9.00,1.65,1.80
max,20.00,6.00,5.00,3.00,4.00,16.00,12.00,11.0,15.00,3.20,3.00


In [10]:
colors = {'background': '#2D0A4E', 'background_2': '#1A0630', 'text': '#F1E9FA', 'muted': '#B9A5D1', 'accent': '#00FF85'}

card_style = {'background': 'rgba(255, 255, 255, 0.06)', 'border': '1px solid rgba(255, 255, 255, 0.12)',
              'borderRadius': 16, 'padding': 20, 'boxShadow': '0 8px 24px rgba(0, 0, 0, 0.35)'}


def kpi_card(title, value, subtitle, color=colors['accent']):
    return html.Div([
        html.Div(title, style={'fontSize': 13, 'color': colors['muted'], 'textTransform': 'uppercase', 'letterSpacing': 1}),
        html.Div(value, style={'fontSize': 30, 'fontWeight': 'bold', 'color': color, 'margin': '6px 0'}),
        html.Div(subtitle, style={'fontSize': 14, 'color': colors['text']})
    ], style={**card_style, 'flex': 1, 'minWidth': 220, 'margin': 8, 'textAlign': 'center',
              'borderTop': f'4px solid {color}'})


def style_fig(fig):
    fig.update_layout(plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', font_color=colors['text'],
                      margin=dict(l=40, r=20, t=50, b=40))
    fig.update_yaxes(gridcolor='rgba(255, 255, 255, 0.1)')
    return fig

In [11]:
def build_dashboard(selected_teams):
    table, scorers = load_data()
    data = table[table["team"].isin(selected_teams)].sort_values("position")

    top = scorers.loc[scorers["goals"].idxmax()] if scorers is not None else None
    leader = table.loc[table["points"].idxmax()]
    attack = table.loc[table["goals_for"].idxmax()]
    defence = table.loc[table["goals_against"].idxmin()]
    most_wins = table.loc[table["won"].idxmax()]
    cards = [
        kpi_card("League leader", leader["team"], f"{leader['points']} points"),
        kpi_card("Best attack", attack["team"], f"{attack['goals_for']} goals scored"),
        kpi_card("Best defence", defence["team"], f"{defence['goals_against']} goals conceded"),
        kpi_card("Top scorer", top["player"], f"{top['team']}: {top['goals']} goals") if scorers is not None
        else kpi_card("Most wins", most_wins["team"], f"{most_wins['won']} wins"),
    ]

    fig_points = style_fig(px.bar(data, x="team", y="points", color="points", title="Points by team"))
    fig_goals = style_fig(px.scatter(data, x="goals_for", y="goals_against", text="team", size="points",
                                     title="Goals scored vs goals conceded"))
    fig_goals.update_traces(textposition="top center")
    fig_bar = style_fig(px.bar(data.sort_values("goals_for", ascending=False), x="team", y="goals_for",
                               title="Goals scored by team"))
    if scorers is not None:
        fig_scorers = style_fig(px.bar(scorers.sort_values("goals").tail(10), x="goals", y="player",
                                       orientation="h", title="Top 10 scorers"))
    else:
        fig_scorers = style_fig(px.bar(data, x="team", y=["won", "drawn", "lost"], title="Wins, draws and losses"))

    shown = data[["position", "team", "played", "won", "drawn", "lost", "goals_for", "goals_against", "goal_diff", "points"]]
    return cards, fig_points, fig_goals, fig_bar, fig_scorers, shown.to_dict("records")


cards, fig_points, fig_goals, fig_bar, fig_scorers, rows = build_dashboard(list(table["team"]))
print([c.children[1].children for c in cards])

['Manchester City', 'Brighton & Hove Albion', 'Leeds United', 'E. Haaland']


In [12]:
table, scorers = load_data()
team_list = sorted(table["team"])

app = Dash(__name__)

app.layout = html.Div(
    style={'minHeight': '100vh', 'padding': '30px 15px', 'fontFamily': 'Segoe UI, Arial, sans-serif',
           'background': f"linear-gradient(135deg, {colors['background']}, {colors['background_2']})"},
    children=[
        html.Div(style={'maxWidth': 1200, 'margin': '0 auto'}, children=[
            html.H1('Premier League Analytics', style={'textAlign': 'center', 'color': colors['accent'], 'margin': 0}),
            html.Div('Scraped from BBC Sport', style={'textAlign': 'center', 'color': colors['muted'], 'marginBottom': 20}),

            html.Label('Teams', style={'color': colors['text']}),
            dcc.Dropdown(team_list, team_list, multi=True, id='team-filter', style={'marginBottom': 15}),

            html.Div(id='kpi-row', style={'display': 'flex', 'flexWrap': 'wrap'}),

            html.Div([
                html.Div(dcc.Graph(id='points-graph'), style={**card_style, 'flex': 1, 'margin': 8}),
                html.Div(dcc.Graph(id='goals-graph'), style={**card_style, 'flex': 1, 'margin': 8}),
            ], style={'display': 'flex', 'flexWrap': 'wrap'}),

            html.Div(dcc.Graph(id='goals-bar'), style={**card_style, 'margin': 8}),
            html.Div(dcc.Graph(id='scorers-graph'), style={**card_style, 'margin': 8}),

            html.Div(id='table-box', style={**card_style, 'margin': 8, 'color': colors['text']}),
        ])
    ])


@app.callback(
    [Output('kpi-row', 'children'), Output('points-graph', 'figure'), Output('goals-graph', 'figure'), Output('goals-bar', 'figure'),
     Output('scorers-graph', 'figure'), Output('table-box', 'children')],
    Input('team-filter', 'value')
)
def update_dashboard(selected_teams):
    cards, fig_points, fig_goals, fig_bar, fig_scorers, rows = build_dashboard(selected_teams or [])
    head = html.Tr([html.Th(col) for col in rows[0].keys()]) if rows else html.Tr()
    body = [html.Tr([html.Td(v) for v in row.values()]) for row in rows]
    return cards, fig_points, fig_goals, fig_bar, fig_scorers, html.Table([head] + body, style={'width': '100%'})


if __name__ == '__main__':
    app.run(debug=True, port=8053, jupyter_mode="external")

Dash app running on http://127.0.0.1:8053/


In [13]:
app = Dash(__name__)

app.layout = html.Div(
    style={'minHeight': '100vh', 'padding': '30px 15px', 'fontFamily': 'Segoe UI, Arial, sans-serif',
           'background': f"linear-gradient(135deg, {colors['background']}, {colors['background_2']})"},
    children=[
        html.Div(style={'maxWidth': 1200, 'margin': '0 auto'}, children=[
            html.H1('Premier League Analytics (live)', style={'textAlign': 'center', 'color': colors['accent'], 'margin': 0}),
            html.Div(id='last-update', style={'textAlign': 'center', 'color': colors['muted'], 'marginBottom': 20}),

            dcc.Interval(id='tick', interval=10 * 60 * 1000, n_intervals=0),

            dcc.Dropdown(team_list, team_list, multi=True, id='team-filter', style={'marginBottom': 15}),
            html.Div(id='kpi-row', style={'display': 'flex', 'flexWrap': 'wrap'}),
            html.Div([
                html.Div(dcc.Graph(id='points-graph'), style={**card_style, 'flex': 1, 'margin': 8}),
                html.Div(dcc.Graph(id='goals-graph'), style={**card_style, 'flex': 1, 'margin': 8}),
            ], style={'display': 'flex', 'flexWrap': 'wrap'}),
            html.Div(dcc.Graph(id='goals-bar'), style={**card_style, 'margin': 8}),
            html.Div(dcc.Graph(id='scorers-graph'), style={**card_style, 'margin': 8}),
        ])
    ])


@app.callback(Output('last-update', 'children'), Input('tick', 'n_intervals'))
def refresh_data(n):
    if n == 0:
        return "Showing the saved data (next refresh in 10 minutes)"
    new_table = scrape_table()
    new_scorers = scrape_scorers()
    if new_table is not None:
        new_table.to_csv("data/pl_table.csv", index=False)
    if new_scorers is not None:
        new_scorers.to_csv("data/pl_scorers.csv", index=False)
    return f"Last refresh: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"


@app.callback(
    [Output('kpi-row', 'children'), Output('points-graph', 'figure'), Output('goals-graph', 'figure'),
     Output('goals-bar', 'figure'), Output('scorers-graph', 'figure')],
    [Input('team-filter', 'value'), Input('last-update', 'children')]
)
def draw_dashboard(selected_teams, last_update):
    cards, fig_points, fig_goals, fig_bar, fig_scorers, rows = build_dashboard(selected_teams or [])
    return cards, fig_points, fig_goals, fig_bar, fig_scorers


if __name__ == '__main__':
    app.run(debug=True, port=8053, jupyter_mode="external")

Dash app running on http://127.0.0.1:8053/
